# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset high-level metadata
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Authors: {getattr(dataset.metadata, 'author', 'N/A')}")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")
print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, their respective fields, columns, and their `@id` values for programmatic reference.

In [ ]:
# List the available record sets in the Croissant schema with their @id

def print_recordset_overview(ds):
    record_sets = ds.metadata.recordSet  # This is a list of RecordSet objects
    if not record_sets:
        print('No record sets defined in the metadata.')
        return []
    print("\nAvailable Record Sets:")
    for i, rs in enumerate(record_sets):
        print(f"{i+1}. @id: {rs['@id']}\tname: {rs.get('name', '-')}")
    return [rs['@id'] for rs in record_sets]

record_set_ids = print_recordset_overview(dataset)

if record_set_ids:
    # For each record set, print its fields (columns)
    for rs in dataset.metadata.recordSet:
        print(f"\nFields for Record Set @id={rs['@id']}:")
        # Fields are under 'field' key
        if 'field' in rs and rs['field']:
            for field in rs['field']:
                if isinstance(field, dict):
                    print(f"- Field @id: {field.get('@id', '[no id]')}, name: {field.get('name', '-')}, dataType: {field.get('dataType', '-')}")
                else:
                    print(f"- Field: {field}")
        else:
            print("  No fields listed.")
else:
    print("No record sets found to explore.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s found above.

> **Note:** Some datasets may have large or numerous record sets. For demonstration, you may wish to extract just a few rows by setting `max_records`.

In [ ]:
# We'll try to extract data from each record set that is available

dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        try:
            print(f"\nLoading first few records from Record Set @id={record_set_id}")
            # Some datasets may have missing implementations or URLs, so wrap with try/except
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Fields (columns): {df.columns.tolist()}")
            print(df.head())
        except Exception as e:
            print(f"  Could not load records for {record_set_id}: {e}")
else:
    print("No record sets present; check the schema or dataset source.")

## 4. Exploratory Data Analysis (EDA)
Apply example data processing steps: filtering records, normalizing numeric fields, detecting/removing outliers, and grouping by category fields. Reference fields using their `@id` in variable assignments.

> If the dataset defines multiple record sets or numeric fields, modify selections below as needed.

In [ ]:
# Try EDA on the first available record set and a numeric column
from pandas.api.types import is_numeric_dtype
import numpy as np

if dataframes:
    # Pick the first loaded record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using Record Set @id: {record_set_id}")

    # Identify a numeric field by checking data types
    numeric_field = None
    for c in df.columns:
        if is_numeric_dtype(df[c]) or np.issubdtype(df[c].dtype, np.number):
            numeric_field = c
            break
        # Try to parse floats if possible
        try:
            df[c] = pd.to_numeric(df[c])
            if is_numeric_dtype(df[c]):
                numeric_field = c
                break
        except:
            continue

    if numeric_field is not None:
        threshold = df[numeric_field].mean()  # Use mean as example filter
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized field '{numeric_field}' (z-score):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a likely categorical field
        group_field = None
        for c in df.columns:
            if c != numeric_field and df[c].nunique() < 12:
                group_field = c
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
            print(grouped_df)
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize a data distribution and relationships between fields in the dataset.

Here are example visualizations for the first available numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    # If there's a group field from above, try a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Boxplot of '{numeric_field}' grouped by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print('Visualization skipped: No numeric data available.')

## 6. Conclusion
In this notebook, you loaded and explored a FAIR mass spectrometry dataset using `mlcroissant`. You reviewed record set and field structure, loaded records, performed example numeric filtering and normalization, and visualized a numeric attribute. For further analysis, consider exploring additional record sets or fields, applying domain-specific feature engineering based on your scientific needs.